In [2]:
# =========================================
# Step RR-1 · Candidate Union (K_cand=200)  —  Patched manifest parser
# =========================================
import os, json, gc
from pathlib import Path
import numpy as np
import pandas as pd

PARQUET_ENGINE = "fastparquet"
TMP_DIR = Path("./tmp").resolve()

# ---- 参数 ----
K_PER_VIEW   = 50           # 输入每图的K
K_CAND       = 200          # 候选上限
ROWS_PER_PART= 10_000       # 每片行数
SEED         = 123

# 输入前缀（对称化 + 行归一 + k=50）
PREF_TAG   = "S_tag_symrow_k50"
PREF_TEXT  = "S_text_symrow_k50"
PREF_BEH   = "S_beh_symrow_k50"
PREF_FUSE  = "S_fused3_symrow_k50"

# 输出前缀
OUT_PREF   = "S_fused3_rr_cand_k200"

np.random.seed(SEED)
print(f"[Env] TMP_DIR = {TMP_DIR}")

# ---------- 工具：稳健的类型转换 ----------
def _to_int(x, default=None):
    """把 x 转成 int；支持 str(含逗号)、float、list(取 len 或 sum)"""
    if x is None:
        return default
    if isinstance(x, (int, np.integer)):
        return int(x)
    if isinstance(x, float):
        return int(x)
    if isinstance(x, str):
        s = x.replace(",", "").replace("_","").strip()
        try:
            return int(float(s))
        except:
            return default
    if isinstance(x, list):
        # 常见两种：parts 为 list → 用 len；nnz 为 list → 用 sum
        # 但这里不知语义，交由调用处决定；这里返回 None 让上层处理
        return None
    return default

def read_manifest(prefix: str):
    man_path = TMP_DIR / f"{prefix}_manifest.json"
    if not man_path.exists():
        # 兼容早期命名
        cands = sorted(TMP_DIR.glob(f"{prefix}*_manifest.json"))
        if not cands:
            raise FileNotFoundError(f"[FATAL] manifest not found for prefix={prefix}")
        man_path = cands[0]

    with open(man_path, "r") as f:
        man = json.load(f)

    # nodes
    nodes = man.get("nodes", man.get("N", man.get("n", None)))
    n_val = _to_int(nodes)
    if n_val is None:
        # 尝试从 index_map 推断
        idmap_path = TMP_DIR / "index_map.parquet"
        if idmap_path.exists():
            id_map = pd.read_parquet(idmap_path, engine=PARQUET_ENGINE, columns=["doc_idx"])
            n_val = int(id_map["doc_idx"].max()) + 1
        else:
            # 最后兜底：从某个 part 文件中取最大 row/col
            part_files = sorted(TMP_DIR.glob(f"{prefix}_part*.parquet"))
            if not part_files:
                raise FileNotFoundError(f"[FATAL] cannot infer nodes, no part files for {prefix}")
            mx = 0
            for fp in part_files[:3]:
                df = pd.read_parquet(fp, engine=PARQUET_ENGINE, columns=["row","col"])
                mx = max(mx, int(df["row"].max()), int(df["col"].max()))
            n_val = mx + 1

    # parts
    parts = man.get("parts", None)
    p_val = _to_int(parts)
    files = man.get("files", None)
    if p_val is None:
        if isinstance(parts, list):
            p_val = len(parts)
        elif isinstance(files, list):
            p_val = len(files)
        else:
            # 扫描磁盘兜底
            p_val = len(sorted(TMP_DIR.glob(f"{prefix}_part*.parquet")))
            if p_val == 0:
                raise FileNotFoundError(f"[FATAL] no part files for {prefix}")

    # nnz（仅打印用，可为 None；若是列表则取 sum）
    nnz = man.get("nnz", man.get("total_edges", None))
    nnz_val = _to_int(nnz)
    if nnz_val is None and isinstance(nnz, list):
        try:
            nnz_val = int(sum([_to_int(x, 0) for x in nnz]))
        except:
            nnz_val = None

    # files（可选）
    if isinstance(files, list) and files:
        file_list = [str((TMP_DIR / fn).name if not str(fn).startswith(str(TMP_DIR)) else Path(fn).name)
                     for fn in files]
    else:
        file_list = []

    return {"nodes": int(n_val), "parts": int(p_val), "nnz": nnz_val, "files": file_list}

def list_part_files(prefix: str, manifest: dict):
    # 优先使用 manifest.files
    if manifest.get("files"):
        fps = [TMP_DIR / fn for fn in manifest["files"]]
        fps = [fp for fp in fps if fp.exists()]
        if fps:
            return fps
    # 否则按规则扫描
    fps = []
    for p in range(manifest["parts"]):
        fp = TMP_DIR / f"{prefix}_part{p:04d}.parquet"
        if fp.exists():
            fps.append(fp)
    if not fps:
        fps = sorted(TMP_DIR.glob(f"{prefix}_part*.parquet"))
    if not fps:
        raise FileNotFoundError(f"[FATAL] no part files for prefix={prefix}")
    return fps

# ---------- 两遍读构建 CSR ----------
def build_csr(prefix: str):
    man = read_manifest(prefix)
    N = man["nodes"]
    part_files = list_part_files(prefix, man)
    print(f"[LOAD] {prefix}: nodes={N:,}, parts={len(part_files)}, nnz≈{man['nnz'] if man['nnz'] else 'unknown'}")

    # pass-1: 计数
    counts = np.zeros(N, dtype=np.int64)
    for i, fp in enumerate(part_files, 1):
        df = pd.read_parquet(fp, engine=PARQUET_ENGINE, columns=["row"])
        r = df["row"].to_numpy(dtype=np.int64, copy=False)
        np.add.at(counts, r, 1)
        if i % 4 == 0 or i == len(part_files):
            print(f"[PASS1] {prefix} parts {i}/{len(part_files)} counted")
        del df
    indptr = np.zeros(N + 1, dtype=np.int64)
    np.cumsum(counts, out=indptr[1:])
    nnz = int(indptr[-1])
    indices = np.empty(nnz, dtype=np.int32)
    values  = np.empty(nnz, dtype=np.float32)
    write_ptr = indptr[:-1].copy()

    # pass-2: 填充
    for i, fp in enumerate(part_files, 1):
        df = pd.read_parquet(fp, engine=PARQUET_ENGINE, columns=["row","col","val"])
        rows = df["row"].to_numpy(np.int64,  copy=False)
        cols = df["col"].to_numpy(np.int32,  copy=False)
        vals = df["val"].to_numpy(np.float32, copy=False)
        order = np.argsort(rows, kind="stable")
        rows, cols, vals = rows[order], cols[order], vals[order]
        if len(rows) > 0:
            uniq_rows, start_idx = np.unique(rows, return_index=True)
            start_idx = np.r_[start_idx, len(rows)]
            for k in range(len(uniq_rows)):
                r = int(uniq_rows[k]); s, e = int(start_idx[k]), int(start_idx[k+1])
                wpos = write_ptr[r]
                ln = e - s
                indices[wpos:wpos+ln] = cols[s:e]
                values [wpos:wpos+ln] = vals[s:e]
                write_ptr[r] += ln
        if i % 4 == 0 or i == len(part_files):
            print(f"[PASS2] {prefix} parts {i}/{len(part_files)} filled")
        del df, rows, cols, vals, order
        gc.collect()
    return N, indptr, indices, values

# ---------- 读取四图为 CSR ----------
N_tag,  ip_tag,  ix_tag,  vx_tag  = build_csr(PREF_TAG)
N_text, ip_text, ix_text, vx_text = build_csr(PREF_TEXT)
N_beh,  ip_beh,  ix_beh,  vx_beh  = build_csr(PREF_BEH)
N_fus,  ip_fus,  ix_fus,  vx_fus  = build_csr(PREF_FUSE)
assert N_tag == N_text == N_beh == N_fus, "[FATAL] node size mismatch among views"
N = N_tag
print(f"[CSR] all views loaded. N={N:,}")

# ---------- 并集 + 标注 + 写分片 ----------
out_parts = 0
rows = np.arange(N, dtype=np.int64)
part_bounds = np.arange(0, N, ROWS_PER_PART, dtype=np.int64)
if part_bounds[-1] != N:
    part_bounds = np.r_[part_bounds, N]

out_files = []
for bi in range(len(part_bounds)-1):
    rs, re = int(part_bounds[bi]), int(part_bounds[bi+1])
    recs = []

    for r in range(rs, re):
        a = ix_tag [ip_tag [r]:ip_tag [r+1]]
        b = ix_text[ip_text[r]:ip_text[r+1]]
        c = ix_beh [ip_beh [r]:ip_beh [r+1]]
        d = ix_fus [ip_fus [r]:ip_fus [r+1]]

        cand = np.unique(np.concatenate((a,b,c,d)))
        if cand.size > K_CAND:
            dv = ix_fus[ip_fus[r]:ip_fus[r+1]]
            dw = vx_fus[ip_fus[r]:ip_fus[r+1]]
            base_score = np.zeros(cand.size, dtype=np.float32)
            if dv.size:
                # 简单线性查找即可；数组很小（<=200）
                for i, cj in enumerate(cand):
                    hit = np.where(dv == cj)[0]
                    base_score[i] = dw[int(hit[0])] if hit.size else 0.0
            take = np.argsort(-base_score, kind="stable")[:K_CAND]
            cand = cand[take]

        aset = set(a.tolist())
        bset = set(b.tolist())
        cset = set(c.tolist())
        dv   = ix_fus[ip_fus[r]:ip_fus[r+1]]
        dw   = vx_fus[ip_fus[r]:ip_fus[r+1]]
        fus_map = {int(dv[i]): float(dw[i]) for i in range(len(dv))}

        for j in cand:
            recs.append((
                int(r), int(j),
                1 if j in aset else 0,
                1 if j in bset else 0,
                1 if j in cset else 0,
                float(fus_map.get(int(j), 0.0))
            ))

    out_df = pd.DataFrame.from_records(
        recs, columns=["row","col","in_tag","in_text","in_beh","fused_base"]
    )
    out_fp = TMP_DIR / f"{OUT_PREF}_part{out_parts:04d}.parquet"
    out_df.to_parquet(out_fp, engine=PARQUET_ENGINE, index=False)
    out_files.append(str(out_fp.name))
    out_parts += 1
    print(f"[SAVE] part#{out_parts-1} rows={len(out_df):,} -> {out_fp.name}")
    del out_df, recs
    gc.collect()

man = {
    "nodes": int(N),
    "K_cand": int(K_CAND),
    "parts": int(out_parts),
    "source": {"tag": PREF_TAG, "text": PREF_TEXT, "beh": PREF_BEH, "fused": PREF_FUSE},
    "files": out_files
}
with open(TMP_DIR / f"{OUT_PREF}_manifest.json", "w") as f:
    json.dump(man, f)
print(f"[MANIFEST] {OUT_PREF}_manifest.json written. parts={out_parts}")
print("[STEP RR-1] DONE: candidate union built.")


[Env] TMP_DIR = /home/koyo/workspace/recsys/tmp
[LOAD] S_tag_symrow_k50: nodes=521,735, parts=15, nnz≈28159756
[PASS1] S_tag_symrow_k50 parts 4/15 counted
[PASS1] S_tag_symrow_k50 parts 8/15 counted
[PASS1] S_tag_symrow_k50 parts 12/15 counted
[PASS1] S_tag_symrow_k50 parts 15/15 counted
[PASS2] S_tag_symrow_k50 parts 4/15 filled
[PASS2] S_tag_symrow_k50 parts 8/15 filled
[PASS2] S_tag_symrow_k50 parts 12/15 filled
[PASS2] S_tag_symrow_k50 parts 15/15 filled
[LOAD] S_text_symrow_k50: nodes=521,735, parts=15, nnz≈28161106
[PASS1] S_text_symrow_k50 parts 4/15 counted
[PASS1] S_text_symrow_k50 parts 8/15 counted
[PASS1] S_text_symrow_k50 parts 12/15 counted
[PASS1] S_text_symrow_k50 parts 15/15 counted
[PASS2] S_text_symrow_k50 parts 4/15 filled
[PASS2] S_text_symrow_k50 parts 8/15 filled
[PASS2] S_text_symrow_k50 parts 12/15 filled
[PASS2] S_text_symrow_k50 parts 15/15 filled
[LOAD] S_beh_symrow_k50: nodes=521,735, parts=14, nnz≈26023186
[PASS1] S_beh_symrow_k50 parts 4/14 counted
[PASS1